<a href="https://colab.research.google.com/github/JosephBigDataAnalytics/JKaremera-Programming-BigDataAnalytics/blob/main/Task_5_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 5 Reflection:

Building the semantic search tool required combining data cleaning, machine learning, and UI design into a single pipeline — the regex extraction carefully stripped URLs and trailing hashtags from each text row, while all-MiniLM-L6-v2 from sentence-transformers converted raw headlines into 384-dimensional embeddings that capture meaning rather than just keywords, meaning a query like "earnings surprise" can surface semantically related results even without an exact word match. A key performance decision was computing all embeddings once at startup and reusing them for every query, keeping cosine similarity lookups near-instant, before wrapping the entire pipeline in a clean Gradio interface that makes the tool accessible to non-technical users through example queries, ranked results, and similarity scores.

Required Task 5

Load the file financial_news.csv.

Last part of the sentence in each row of text contains an url. Remove this from text and create new column called URL and add the url.
Create sentence embeddings for the modified column text. Using Gradio, build a semantic search tool where the user enters some text (such as (“earnings surprise”, “regulatory fine”), and the top 5 closest records (based on cosine similarity) are output to the user.

# Semantic Search Tool for Financial News using Gradio

In [ ]:
# --- Imports ---
# pandas        → data loading and manipulation
# re            → regular expressions for URL extraction
# sentence-transformers → pre-trained model to create sentence embeddings
# sklearn       → cosine similarity computation
# gradio        → build the interactive web UI
# numpy         → array operations

import pandas as pd
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

# STEP 1: Load the CSV File

In [ ]:
# The file has two columns: 'text' and 'label'.
# Each row in 'text' ends with a URL (e.g. https://t.co/xxxxxxx)
# sometimes followed by hashtags (e.g. #trading #markets).

df = pd.read_csv('/content/financial_news.csv')

print(f"Loaded {len(df)} rows")
print(df.head(3))
print(df.columns.tolist())

Loaded 16990 rows
                                                text  label
0  Here are Thursday's biggest analyst calls: App...      0
1  Buy Las Vegas Sands as travel to Singapore bui...      0
2  Piper Sandler downgrades DocuSign to sell, cit...      0
['text', 'label']


# STEP 2: Extract URLs and Clean the 'text' Column

In [ ]:
# Strategy:
#   - Use a regex pattern to find the URL in each text string.
#     The pattern r'https?://\S+' matches any URL starting with
#     http:// or https:// up to the first whitespace.
#   - Extract the URL into a new column 'URL'.
#   - Remove the URL (and any trailing hashtags/whitespace) from 'text'.
#
# re.search() → finds the FIRST match of the pattern in the string.
# group(0)    → returns the matched string itself.
# re.sub()    → replaces the matched pattern with an empty string.

def extract_url(text):
    """Extract the first URL found in the text string."""
    match = re.search(r'https?://\S+', str(text))
    return match.group(0) if match else None

def clean_text(text):
    """
    Remove the URL and any trailing hashtags/whitespace from the text.
    Step 1: Remove everything from the URL onwards (URL + hashtags).
    Step 2: Strip leading/trailing whitespace.
    """
    # Remove URL and anything after it (hashtags, extra spaces)
    cleaned = re.sub(r'https?://\S+.*', '', str(text))
    # Strip residual whitespace and punctuation from the end
    return cleaned.strip().rstrip('.,;:')

# Apply both functions to create the new columns
df['URL']  = df['text'].apply(extract_url)
df['text'] = df['text'].apply(clean_text)

print("\nAfter URL extraction:")
print(df[['text', 'URL', 'label']].head(5))


After URL extraction:
                                                text                      URL  \
0  Here are Thursday's biggest analyst calls: App...  https://t.co/QPN8Gwl7Uh   
1  Buy Las Vegas Sands as travel to Singapore bui...  https://t.co/fLS2w57iCz   
2  Piper Sandler downgrades DocuSign to sell, cit...  https://t.co/1EmtywmYpr   
3  Analysts react to Tesla's latest earnings, bre...  https://t.co/kwhoE6W06u   
4  Netflix and its peers are set for a ‘return to...  https://t.co/jPpdl0D9s4   

   label  
0      0  
1      0  
2      0  
3      0  
4      0  


# STEP 3: Drop any Rows with Empty Text

In [ ]:
# Some rows may have had only a URL or be completely empty after
# cleaning. Remove them to avoid embedding blank strings.

df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)
print(f"\nRows after cleaning: {len(df)}")


Rows after cleaning: 16976


# STEP 4: Create Sentence Embeddings

In [ ]:
# We use the 'all-MiniLM-L6-v2' model from sentence-transformers.
# This is a lightweight, fast model that maps each sentence to a
# 384-dimensional dense vector (embedding).
#
# Why embeddings?
#   Raw text can't be compared mathematically. Embeddings convert
#   sentences into vectors where semantically similar sentences
#   are close together in vector space.
#
# model.encode() processes all texts at once (batch mode).
# show_progress_bar=True gives a live progress indicator.

print("\nLoading sentence transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Creating embeddings for all news headlines...")
embeddings = model.encode(
    df['text'].tolist(),
    show_progress_bar=True,
    batch_size=64        # process 64 sentences at a time for efficiency
)

print(f"\nEmbeddings shape: {embeddings.shape}")
# Expected: (number_of_rows, 384)



Loading sentence transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings for all news headlines...


Batches:   0%|          | 0/266 [00:00<?, ?it/s]


Embeddings shape: (16976, 384)


# STEP 5: Define the Semantic Search Function

In [ ]:
# This function is called every time the user submits a query
# in the Gradio UI.
#
# How it works:
#   1. Encode the user's query string into a vector using the same model.
#   2. Compute cosine similarity between the query vector and ALL
#      news headline embeddings.
#      - Cosine similarity = 1 → identical meaning
#      - Cosine similarity = 0 → completely unrelated
#   3. Sort by similarity (descending) and return the top 5 results.
#   4. Return as a formatted pandas DataFrame for display.

def semantic_search(query: str, top_k: int = 5):
    """
    Search the financial news dataset semantically.

    Args:
        query  : The user's natural language search query.
        top_k  : Number of top results to return (default 5).

    Returns:
        A pandas DataFrame with columns: Rank, Headline, Label, URL, Similarity.
    """
    if not query.strip():
        return pd.DataFrame({"Message": ["Please enter a search query."]})

    # Step 5a: Encode the query into a vector
    query_embedding = model.encode([query])  # shape: (1, 384)

    # Step 5b: Compute cosine similarity between query and all embeddings
    # cosine_similarity returns a (1, N) array — squeeze to 1D
    similarities = cosine_similarity(query_embedding, embeddings)[0]  # shape: (N,)

    # Step 5c: Get indices of top_k most similar results
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Step 5d: Build the results DataFrame
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "Rank"       : rank,
            "Headline"   : df['text'].iloc[idx],
            "Label"      : df['label'].iloc[idx],
            "URL"        : df['URL'].iloc[idx],
            "Similarity" : f"{similarities[idx]:.4f}"
        })

    return pd.DataFrame(results)

# STEP 6: Build the Gradio Interface

In [ ]:
# Gradio lets us wrap the search function in a web UI with
# no front-end code required.
#
# gr.Interface() wires together:
#   fn         → the Python function to call on submit
#   inputs     → a text box for the query
#   outputs    → a DataFrame table to display results
#   examples   → pre-loaded example queries to help the user get started
#
# launch() starts a local web server (opens in browser automatically).

print("\nLaunching Gradio interface...")

interface = gr.Interface(
    fn=semantic_search,

    inputs=gr.Textbox(
        label="Search Query",
        placeholder='e.g. "earnings surprise" or "regulatory fine" or "buy rating upgrade"',
        lines=2
    ),

    outputs=gr.Dataframe(
        label="Top 5 Matching Financial News Headlines",
        headers=["Rank", "Headline", "Label", "URL", "Similarity"],
        wrap=True           # wrap long text so it's readable in the table
    ),

    title="📈 Financial News Semantic Search",
    description=(
        "Enter any financial topic or phrase and retrieve the 5 most "
        "semantically similar news headlines from the dataset. "
        "Results are ranked by cosine similarity score (1.0 = perfect match)."
    ),

    examples=[
        ["earnings surprise"],
        ["regulatory fine"],
        ["stock upgrade buy rating"],
        ["interest rate recession"],
        ["CEO resignation leadership change"],
    ],

    allow_flagging="never"   # hides the 'flag' button for a cleaner UI
)

# share=False → local only (set share=True to get a public URL)
interface.launch(share=False)


Launching Gradio interface...


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>